# Manual per-mixture baseline deconvolution

Reconstruct the exact reads sampled for each stored pseudobulk mixture, index a mixture,
and run the three baselines (`uxm`, `celfie`, `celfieish`) on it — the same code path the
pipeline uses (`prepare=False`, reads already carry atlas-region `name`s).

**Heads-up on your file:** `tmp/pseudobulk_generation/pseudobulk.h5` contains only
`pure_profiles` — its `outputs/<split>/pseudobulks` groups are **empty** (0 mixtures).
Point `H5_PATH` below at a run that actually generated mixtures.

**Genome/atlas must match the reads.** The reads in the default file are **hg38**
(`name` like `chr11:75825997-75826219`), so use hg38 atlases. With `prepare=False` the
deconvolvers group reads by `name`, so atlas region names must match — the diagnostic cell
reports the overlap.

In [1]:
import os, sys

REPO_ROOT = "/home/luna.kuleuven.be/u0169940/Repos/syto"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)  # makes relative atlas paths resolve

# --- Data ------------------------------------------------------------------
# The tmp file you named has NO mixtures. This 100k-per-split run does:
H5_PATH = "/mnt/data/syto_experiments/mlflow/198121009408531961/a3f1f111ffc14085bc70c8bf60a78e4d/artifacts/pseudobulk_generation/pseudobulk.h5"
SPLIT = "test"
CLASS_LABEL_COLUMN = "original_label"

# --- Atlases (MUST match the genome build / region naming of the reads) -----
# UXM_ATLAS_PATH = "/home/luna.kuleuven.be/u0169940/Repos/UXM_deconv/supplemental/Atlas.U25.l4.hg38.full.tsv"
UXM_ATLAS_PATH = "/mnt/data/loyfer/atlases/U25.l4.hg38_trainonly_uxm_atlas.csv"
UXM_IGNORE_CELLS = ["Megakaryocytes"]

# CELFIE_ATLAS_PATH = "/mnt/data/loyfer/atlases/U25.l4.hg38.BetaCountsMethylAtlas.csv"
CELFIE_ATLAS_PATH = "/mnt/data/loyfer/atlases/U25.l4.hg38.trainonly.BetaCountsMethylAtlas.csv"
CELFIE_REFERENCE_GENOME = "hg38"

# --- EM knobs (convergence mode -> one proportion vector per model) ---------
CELFIE_KW = dict(
    num_iterations=10000,
    convergence_criteria=1e-4,
    random_restarts=10,
    sum_by_region=True,   # <- flip to True to pool CpGs per region (CelFiE convention)
    freeze_gamma=True,    # <- flip to True to hold the atlas methylation fixed
)

CELFIE_KW_gamma = dict(
    num_iterations=1000,
    convergence_criteria=1e-4,
    random_restarts=10,
    sum_by_region=True,   # <- flip to True to pool CpGs per region (CelFiE convention)
    freeze_gamma=False,    # <- flip to True to hold the atlas methylation fixed
)

CELFIE_KW_nosum = dict(
    num_iterations=1000,
    convergence_criteria=1e-4,
    random_restarts=10,
    sum_by_region=False,   # <- flip to True to pool CpGs per region (CelFiE convention)
    freeze_gamma=True,    # <- flip to True to hold the atlas methylation fixed
)

CELFIE_KW_baseline = dict(
    num_iterations=10000,
    convergence_criteria=1e-3,
    random_restarts=1,
    sum_by_region=False,   # <- flip to True to pool CpGs per region (CelFiE convention)
    freeze_gamma=False,    # <- flip to True to hold the atlas methylation fixed
)

CELFIEISH_KW = dict(num_iterations=1000, convergence_criteria=1e-4)

In [2]:
import h5py
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from syto.data.pseudobulk_hdf5_utils import PseudobulkHDF5Reader
from syto.data.pseudobulk_generator import _sample_read_ids_from_grouped_dataframe

# Label mapping straight from the file's parameters (no external json needed).
with h5py.File(H5_PATH, "r") as f:
    _ct = f["parameters/cell_types_mapping"][...]
labels_dict = {int(i): n.decode() for n, i in _ct}
labels_dict_reversed = {v: k for k, v in labels_dict.items()}
n_labels = len(labels_dict)
print(n_labels, "cell types")

39 cell types


In [ ]:
# Shared reconstruction state (loaded once) + per-mixture params.
reader = PseudobulkHDF5Reader(H5_PATH)
input_df, idx_map = reader.build_reconstruction_state(SPLIT, class_label_column=CLASS_LABEL_COLUMN)
mix_params = list(reader.iter_pseudobulk_params(SPLIT))  # [(seed, n_reads_per_gr, target_proportions), ...]
print(f"{len(mix_params):,} mixtures in split '{SPLIT}' | input reads: {len(input_df):,}")


def get_mixture(i):
    """Reconstruct the exact reads + ground-truth proportions for mixture i."""
    seed, n_reads, target = mix_params[i]
    read_ids = _sample_read_ids_from_grouped_dataframe(n_reads, idx_map, seed=seed)
    reads = input_df.iloc[read_ids].reset_index(drop=True)
    return reads, np.asarray(target, dtype=float)

In [25]:
# Build the three deconvolvers once (atlas loading is the slow part).
from syto.data.atlases.uxm_atlases import UXMMethylationAtlas
from syto.data.atlases.celfieish_atlases import CpGBetaCountsMethylationAtlas
from baselines.deconvolution.uxm.uxm import UXMDeconvolver, mark_records_methyl_state
from baselines.deconvolution.celfie.celfie import CelFiEDeconvolver
from baselines.deconvolution.celfieish.celfieish import CelFiEISHDeconvolver
from baselines.deconvolution.epidish.epidish import EpiDishDeconvolver

uxm_atlas = UXMMethylationAtlas(
    atlas_name="uxm", reference_genome="hg38",
    atlas_path=UXM_ATLAS_PATH, ignore=UXM_IGNORE_CELLS,
)
beta_atlas = CpGBetaCountsMethylationAtlas(
    atlas_name="celfie", reference_genome=CELFIE_REFERENCE_GENOME,
    atlas_path=CELFIE_ATLAS_PATH,
)

deconvolvers = {
    "uxm": UXMDeconvolver(uxm_atlas),
    "celfie_proposed": CelFiEDeconvolver(beta_atlas, **CELFIE_KW),
    "epidish": EpiDishDeconvolver(beta_atlas),
    "epidish_houseman": EpiDishDeconvolver(beta_atlas, method="CP"),
    "celfieish": CelFiEISHDeconvolver(beta_atlas, **CELFIEISH_KW),
}
for name, d in deconvolvers.items():
    print(f"{name:10} -> {len(d.atlas.ref_cells)} ref cells")

uxm        -> 39 ref cells
celfie_proposed -> 39 ref cells
epidish    -> 39 ref cells
epidish_houseman -> 39 ref cells
celfieish  -> 39 ref cells


In [5]:
import datetime

In [6]:
def run_all(reads):
    """Run every baseline on one mixture's reads (mirrors the pipeline: prepare=False)."""
    print(datetime.datetime.now(), "Start deconvolution")
    out = {}
    for name, d in deconvolvers.items():
        r = reads.copy()
        if name == "uxm":
            r = mark_records_methyl_state(r)  # UXM needs M/U/X columns
        
        res = d.deconvolute_reads(r, labels_dict_reversed, n_labels=n_labels, prepare=False)
        out[name] = None if res is None else np.asarray(res, dtype=float)
        print(datetime.datetime.now(), f"{name} is done")
    return out


def _atlas_region_names(d):
    try:
        return set(d.atlas.atlas["name"])
    except Exception:
        return None


def region_overlap(reads):
    """How many of this mixture's regions each atlas actually knows about.
    If regions_in_atlas is ~0 the genome/atlas doesn't match -> results will be junk/None."""
    names = set(reads["name"].unique())
    rows = []
    for mn, d in deconvolvers.items():
        aset = _atlas_region_names(d)
        rows.append((mn, len(names), None if aset is None else len(names & aset)))
    return pd.DataFrame(rows, columns=["model", "mixture_regions", "regions_in_atlas"])

## Inspect a single mixture
Change `INDEX` and re-run.

In [7]:
INDEX = 0

In [22]:
INDEX += 1

reads, target = get_mixture(INDEX)
print(f"mixture {INDEX}: {len(reads):,} reads | nonzero target classes: {int((target > 0).sum())} | target sum: {target.sum():.3f}")
region_overlap(reads)

for i in range(39):
    target_value = target[i]
    if target_value>0:
        print(labels_dict[i],":", target_value)

mixture 9: 474,999 reads | nonzero target classes: 3 | target sum: 1.000
Epid-Kerat : 0.12544536824620167
Lung-Ep-Bron : 0.7314042946377904
Oligodend : 0.14315033711600783


In [26]:
preds = run_all(reads)  # {'uxm': array, 'celfie': array, 'celfieish': array}

comp = pd.DataFrame({"cell_type": [labels_dict[i] for i in range(n_labels)], "target": target})
for name, p in preds.items():
    comp[name] = p if p is not None else np.nan

print(comp.sort_values("target", ascending=False).head(15).reset_index(drop=True))

2026-07-24 21:17:51.488930 Start deconvolution
2026-07-24 21:17:52.264215 uxm is done
2026-07-24 21:17:59.439633 celfie_proposed is done
2026-07-24 21:18:02.540680 epidish is done
2026-07-24 21:18:05.681555 epidish_houseman is done
2026-07-24 21:18:34.129646 celfieish is done
            cell_type    target       uxm  celfie_proposed   epidish  \
0        Lung-Ep-Bron  0.731404  0.705975         0.672544  0.601072   
1           Oligodend  0.143150  0.137239         0.134758  0.138639   
2          Epid-Kerat  0.125445  0.106093         0.104351  0.089464   
3        Blood-Granul  0.000000  0.000000         0.000765  0.001584   
4          Adipocytes  0.000000  0.007202         0.001040  0.002582   
5          Bladder-Ep  0.000000  0.000000         0.004804  0.004471   
6             Blood-B  0.000000  0.000000         0.000095  0.000000   
7             Blood-T  0.000000  0.000000         0.000313  0.002453   
8            Blood-NK  0.000000  0.007739         0.001693  0.002405   
9  

In [23]:
preds = run_all(reads)  # {'uxm': array, 'celfie': array, 'celfieish': array}

comp = pd.DataFrame({"cell_type": [labels_dict[i] for i in range(n_labels)], "target": target})
for name, p in preds.items():
    comp[name] = p if p is not None else np.nan

print(comp.sort_values("target", ascending=False).head(15).reset_index(drop=True))

2026-07-24 21:15:44.915898 Start deconvolution
2026-07-24 21:15:45.706966 uxm is done
2026-07-24 21:15:53.201662 celfie_proposed is done
2026-07-24 21:15:56.314334 epidish is done
2026-07-24 21:15:59.413894 epidish_houseman is done
2026-07-24 21:16:21.390054 celfieish is done
            cell_type    target       uxm  celfie_proposed   epidish  \
0        Lung-Ep-Bron  0.731404  0.721984         0.714430  0.693391   
1           Oligodend  0.143150  0.140969         0.144111  0.138728   
2          Epid-Kerat  0.125445  0.102520         0.101329  0.103939   
3        Blood-Granul  0.000000  0.000000         0.000669  0.000000   
4          Adipocytes  0.000000  0.000000         0.000769  0.001265   
5          Bladder-Ep  0.000000  0.000000         0.000721  0.000000   
6             Blood-B  0.000000  0.000000         0.000129  0.000000   
7             Blood-T  0.000000  0.000000         0.001837  0.000000   
8            Blood-NK  0.000000  0.006547         0.000219  0.003328   
9  

In [10]:
print(comp.sort_values("target", ascending=False).head(15).reset_index(drop=True))

            cell_type  target   uxm  celfie_proposed  epidish  \
0         Smooth-Musc   1.000 0.794            0.910    0.703   
1          Bladder-Ep   0.000 0.006            0.000    0.004   
2          Adipocytes   0.000 0.000            0.000    0.000   
3        Blood-Granul   0.000 0.000            0.000    0.000   
4    Blood-Mono+Macro   0.000 0.000            0.000    0.011   
5            Blood-NK   0.000 0.000            0.000    0.000   
6             Blood-B   0.000 0.014            0.000    0.000   
7             Blood-T   0.000 0.007            0.000    0.019   
8         Bone-Osteob   0.000 0.000            0.011    0.027   
9   Breast-Luminal-Ep   0.000 0.000            0.000    0.008   
10    Breast-Basal-Ep   0.000 0.000            0.000    0.000   
11        Colon-Fibro   0.000 0.000            0.000    0.036   
12       Dermal-Fibro   0.000 0.000            0.005    0.030   
13           Endothel   0.000 0.000            0.000    0.011   
14           Colon-Ep   0

In [9]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [ ]:
def metrics(target, pred):
    if pred is None:
        return {"R2": np.nan, "L1": np.nan, "cos": np.nan}
    ss_res = np.sum((target - pred) ** 2)
    ss_tot = np.sum((target - target.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    denom = np.linalg.norm(target) * np.linalg.norm(pred)
    cos = float(target @ pred / denom) if denom > 0 else np.nan
    return {"R2": r2, "L1": float(np.abs(target - pred).sum()), "cos": cos}


pd.DataFrame({name: metrics(target, p) for name, p in preds.items()}).T

## Aggregate over several mixtures
Replicates the flattened-across-mixtures R² you compared. Keep `K` small — each mixture can
carry hundreds of thousands of reads, so this is the slow cell.

In [ ]:
K = 20

targets, preds_by_model = [], {m: [] for m in deconvolvers}
for i in tqdm(range(K)):
    r, t = get_mixture(i)
    out = run_all(r)
    targets.append(t)
    for m, p in out.items():
        preds_by_model[m].append(p if p is not None else np.full(n_labels, np.nan))

T = np.vstack(targets)
summary = {}
for m in deconvolvers:
    P = np.vstack(preds_by_model[m])
    ok = ~np.isnan(P).any(axis=1)
    ss_res = np.sum((T[ok] - P[ok]) ** 2)
    ss_tot = np.sum((T[ok] - T[ok].mean()) ** 2)
    summary[m] = {"n_mixtures": int(ok.sum()), "R2_flat": 1 - ss_res / ss_tot,
                  "mean_L1": float(np.abs(T[ok] - P[ok]).sum(axis=1).mean())}
pd.DataFrame(summary).T